# exp-001 tfidf-regex-lgbm-mae — text-only price regression

In [ ]:
import os
os.environ["PYTHONHASHSEED"] = "42"
import json
import random
import re
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import cloudpickle
import lightgbm as lgb
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.model_selection import KFold


In [ ]:
PROJECT_ROOT = "../../.."
EXP_DIR = "."
SEED = 42
N_SPLITS = 5
N_THREADS = 4
TFIDF_MAX_FEATURES = 6000
TFIDF_MIN_DF = 3
SVD_COMPONENTS = 60
N_ESTIMATORS = 300
LEARNING_RATE = 0.05
NUM_LEAVES = 31


In [ ]:
random.seed(SEED)
np.random.seed(SEED)

sys.path.insert(0, str(Path(PROJECT_ROOT).resolve()))
from tools.config import load_config, resolve_metric

root = Path(PROJECT_ROOT).resolve()
exp_dir = Path(EXP_DIR).resolve()
cfg = load_config(root)
mae = resolve_metric(cfg["task"]["metric"], root)


## Load

In [ ]:
raw = root / cfg["paths"]["raw"]
train = pd.read_csv(raw / "train.csv")
test = pd.read_csv(raw / "test.csv")
print(train.shape, test.shape)


## Feature engineering: regex specs + TF-IDF (word + char) + SVD

`REGEX_PATTERNS` covers the amenity/spec mentions found during EDA (bedroom, bathroom, sqft, acreage, garage, pool, HOA, renovation, new construction) plus the `[Redacted Entity]` placeholder token. `TextPriceModel` bundles both TF-IDF vectorizers, both `TruncatedSVD` reducers, and the LightGBM regressor into one object so the same class definition serializes with the model (`cloudpickle`, defined here in `__main__`) and can be refit per CV fold without leaking fold-validation vocabulary into training.

In [ ]:
REGEX_PATTERNS = {
    "bedroom": r"\b\d+\s*(?:-|\s)?bed(?:room)?s?\b",
    "bathroom": r"\b\d+(?:\.\d+)?\s*(?:-|\s)?bath(?:room)?s?\b",
    "sqft": r"\b[\d,]+\s*(?:sq\.?\s?ft|square\s?feet|sqft)\b",
    "acre": r"\b\d+(?:\.\d+)?\s*acres?\b",
    "year_built": r"\bbuilt\s+in\s+\d{4}\b",
    "garage": r"\bgarage\b",
    "pool": r"\bpool\b",
    "hoa": r"\bHOA\b",
    "renovated": r"\brenovat\w*\b",
    "new_construction": r"\bnew construction\b",
    "redacted": r"\[Redacted Entity\]",
}


def regex_features(texts):
    texts = texts.fillna("")
    feats = {}
    for name, pat in REGEX_PATTERNS.items():
        feats[f"has_{name}"] = texts.str.contains(
            pat, regex=True, case=False, na=False
        ).astype(np.float64)
    feats["char_len"] = texts.str.len().astype(np.float64)
    feats["word_count"] = texts.str.split().str.len().astype(np.float64)
    return pd.DataFrame(feats).values


class TextPriceModel:
    def __init__(self, seed, n_threads, tfidf_max_features, tfidf_min_df,
                 svd_components, n_estimators, learning_rate, num_leaves):
        self.seed = seed
        self.n_threads = n_threads
        self.tfidf_max_features = tfidf_max_features
        self.tfidf_min_df = tfidf_min_df
        self.svd_components = svd_components
        self.n_estimators = n_estimators
        self.learning_rate = learning_rate
        self.num_leaves = num_leaves

    def _feature_matrix(self, texts, fit):
        texts = texts.fillna("")
        if fit:
            self.word_tfidf = TfidfVectorizer(
                ngram_range=(1, 2), max_features=self.tfidf_max_features,
                min_df=self.tfidf_min_df, sublinear_tf=True)
            self.char_tfidf = TfidfVectorizer(
                analyzer="char_wb", ngram_range=(3, 5),
                max_features=self.tfidf_max_features,
                min_df=self.tfidf_min_df, sublinear_tf=True)
            word_sparse = self.word_tfidf.fit_transform(texts)
            char_sparse = self.char_tfidf.fit_transform(texts)
            self.word_svd = TruncatedSVD(
                n_components=self.svd_components, random_state=self.seed)
            self.char_svd = TruncatedSVD(
                n_components=self.svd_components, random_state=self.seed)
            word_dense = self.word_svd.fit_transform(word_sparse)
            char_dense = self.char_svd.fit_transform(char_sparse)
        else:
            word_dense = self.word_svd.transform(self.word_tfidf.transform(texts))
            char_dense = self.char_svd.transform(self.char_tfidf.transform(texts))
        return np.hstack([word_dense, char_dense, regex_features(texts)])

    def fit(self, texts, y):
        X = self._feature_matrix(texts, fit=True)
        self.model = lgb.LGBMRegressor(
            objective="regression_l1",
            n_estimators=self.n_estimators,
            learning_rate=self.learning_rate,
            num_leaves=self.num_leaves,
            feature_fraction=1.0,
            bagging_fraction=1.0,
            random_state=self.seed,
            deterministic=True,
            force_row_wise=True,
            num_threads=self.n_threads,
            verbose=-1,
        )
        self.model.fit(X, y)
        return self

    def predict_texts(self, texts):
        X = self._feature_matrix(texts, fit=False)
        return np.clip(self.model.predict(X), 0, None)

    def predict(self, **frames):
        test_df = frames["test"]
        preds = self.predict_texts(test_df["text"])
        return pd.DataFrame({"id": test_df["id"], "listPrice": preds})


def make_model():
    return TextPriceModel(
        seed=SEED, n_threads=N_THREADS,
        tfidf_max_features=TFIDF_MAX_FEATURES, tfidf_min_df=TFIDF_MIN_DF,
        svd_components=SVD_COMPONENTS, n_estimators=N_ESTIMATORS,
        learning_rate=LEARNING_RATE, num_leaves=NUM_LEAVES,
    )


## Cross-validation

TF-IDF and `TruncatedSVD` are refit inside every fold (via `TextPriceModel.fit`), so no vocabulary from the held-out fold ever reaches the training-fold transformers.

In [ ]:
kf = KFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)
y = train["listPrice"].values
texts = train["text"]

fold_scores = []
for fold, (tr_idx, va_idx) in enumerate(kf.split(texts)):
    fold_model = make_model()
    fold_model.fit(texts.iloc[tr_idx], y[tr_idx])
    fold_pred = fold_model.predict_texts(texts.iloc[va_idx])
    fold_mae = mae(y[va_idx], fold_pred)
    fold_scores.append(fold_mae)
    print(f"fold {fold}: mae={fold_mae:.2f}")

cv_primary = float(np.mean(fold_scores))
cv_std = float(np.std(fold_scores))
print(f"cv_primary={cv_primary:.2f} cv_std={cv_std:.2f}")


## Save & Load Model

The final bundle refits `TextPriceModel` on the full `train.csv` (all 14,640 rows) so the vectorizers see the complete training vocabulary before scoring `test.csv`.

In [ ]:
final_model = make_model()
final_model.fit(texts, y)

with open(exp_dir / "model.pkl", "wb") as fh:
    cloudpickle.dump(final_model, fh)


In [ ]:
with open(exp_dir / "model.pkl", "rb") as fh:
    loaded = cloudpickle.load(fh)


## Prediction

In [ ]:
pred = final_model.predict(test=test)
pred.to_csv(exp_dir / "submission.csv", index=False,
            float_format="%.6f", lineterminator="\n")

pred_loaded = loaded.predict(test=test)
pred_loaded.to_csv(exp_dir / "submission_check.csv", index=False,
                    float_format="%.6f", lineterminator="\n")
assert (pred["listPrice"] == pred_loaded["listPrice"]).all()

metrics_out = {
    "cv_primary": cv_primary,
    "cv_std": cv_std,
    "fold_scores": fold_scores,
    "parent_cv_primary": 550252.456284153,
    "delta_vs_parent": 550252.456284153 - cv_primary,
}
print(json.dumps(metrics_out, indent=2))
with open(exp_dir / "cv_metrics.json", "w") as fh:
    json.dump(metrics_out, fh, indent=2)
